## 1. Imports & Environment Setup

In [ ]:
# ── Suppress verbose TensorFlow logs ──────────────────────
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'          # Suppress TF C++ logs
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'         # Suppress oneDNN notice

# ── Core imports ──────────────────────────────────────────
import zipfile
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout, GlobalAveragePooling2D, SeparableConv2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}  ← CPU-only mode')

# Limit TF to avoid OOM on capped 1024MB VM
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(2)
print('\n✅ All imports successful — running in CPU-only mode')

## 2. Project Directory Structure

In [ ]:
# ── Define and create project folder structure ───────────
BASE_DIR    = 'vehicle_detection_project'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
IMAGES_DIR  = os.path.join(DATA_DIR, 'images')
MODELS_DIR  = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

for folder in [BASE_DIR, DATA_DIR, IMAGES_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('\nFolder structure:')
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')


## 3. Data Loading

### 3.1 Extract Images from ZIP

In [ ]:
# ─────────────────────────────────────────────────────────
# UPDATE this path to match where images.zip is stored
# on the Simplilearn VM (check your LMS download folder)
# ─────────────────────────────────────────────────────────
ZIP_PATH = '/voc/work/Images.zip'   # ← CHANGE if needed

if os.path.exists(ZIP_PATH):
    print(f'Found: {ZIP_PATH}')
    print('Extracting... (large files may take a few minutes)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(IMAGES_DIR)
    print('✅ Extraction complete!')
else:
    print(f'❌ File not found: {ZIP_PATH}')
    print('   → Update ZIP_PATH to the correct location')

# Auto-detect image extension used in this dataset
all_imgs = []
for ext in ['*.jpg', '*.jpeg', '*.png']:
    import glob
    all_imgs.extend(glob.glob(os.path.join(IMAGES_DIR, '**', ext), recursive=True))
    all_imgs.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))

all_imgs = list(set(all_imgs))
print(f'\nTotal images found: {len(all_imgs)}')
if all_imgs:
    print(f'Sample path: {all_imgs[0]}')

### 3.2 Load Annotations (labels.csv)

In [ ]:
# ─────────────────────────────────────────────────────────
# labels.csv format (no header):
#   image_id, class_label, xmin, ymin, xmax, ymax
# e.g.: 00000000,car,194,78,273,122
# ─────────────────────────────────────────────────────────
LABELS_CSV = '/voc/work/labels.csv'   # ← CHANGE if needed

df = pd.read_csv(
    LABELS_CSV,
    header=None,
    names=['image_id', 'label', 'xmin', 'ymin', 'xmax', 'ymax']
)

print(f'Total annotation rows : {len(df):,}')
print(f'Unique images         : {df["image_id"].nunique():,}')
print(f'Unique classes        : {df["label"].nunique()}')
print()
print('Class distribution:')
print(df['label'].value_counts().to_string())
print()
print('Sample rows:')
display(df.head(8))

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── Visualize class distribution ─────────────────────────
class_counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
class_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Annotation Count per Vehicle Class', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Vehicle Class')
axes[0].set_ylabel('Number of Annotations')
axes[0].tick_params(axis='x', rotation=35)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', startangle=140, shadow=True)
axes[1].set_title('Class Share (All Annotations)', fontweight='bold', fontsize=13)

plt.suptitle('Vehicle Class Distribution in labels.csv', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '01_class_distribution.png'), dpi=120)
plt.show()

## 5. Data Preprocessing

### 5.1 Deduplicate & Filter to Available Images

Keep only the dominant bounding box per image and cross-match with images physically present on disk.

In [ ]:
import glob

# ── Configuration ────────────────────────────────────────
IMG_SIZE   = 64       # 64×64 — low memory footprint
MAX_IMAGES = 5000     # Cap to stay within 1024 MB RAM
BATCH_SIZE = 32
EPOCHS     = 30

# ── Fix IMAGES_DIR to actual extracted subfolder ─────────
IMAGES_DIR = 'vehicle_detection_project/data/images/Images'

# ── Get image IDs physically available on disk ───────────
available_files = os.listdir(IMAGES_DIR)
available_ids   = set(f.replace('.jpg', '') for f in available_files if f.endswith('.jpg'))
print(f'Images physically available : {len(available_ids)}')

# ── Filter labels to matched images only ─────────────────
df['image_id_str'] = df['image_id'].astype(str).str.zfill(8)
df_available = df[df['image_id_str'].isin(available_ids)].copy()
print(f'Label rows matching folder  : {len(df_available)}')
print(f'Unique matched images       : {df_available["image_id_str"].nunique()}')

# ── One dominant box per image (largest area) ─────────────
df_available['area'] = (
    (df_available['xmax'] - df_available['xmin']) *
    (df_available['ymax'] - df_available['ymin'])
)
df_single = df_available.loc[
    df_available.groupby('image_id_str')['area'].idxmax()
].reset_index(drop=True)
print(f'Unique images after dedup   : {len(df_single)}')

# ── Subsample if dataset exceeds memory limit ────────────
from sklearn.model_selection import train_test_split
if len(df_single) > MAX_IMAGES:
    df_single, _ = train_test_split(
        df_single, train_size=MAX_IMAGES,
        stratify=df_single['label'], random_state=42
    )
    print(f'✂️  Subsampled to {MAX_IMAGES} images (memory limit)')

# ── Encode class labels ───────────────────────────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_single['label_enc'] = le.fit_transform(df_single['label'])
NUM_CLASSES = len(le.classes_)
class_names = list(le.classes_)
print(f'\nClasses ({NUM_CLASSES}): {class_names}')


### 5.2 Load Images into Memory

In [ ]:
# ── Load all matched images into memory ──────────────────
X_images, y_labels, y_bboxes = [], [], []
skipped = 0

print(f'Loading {len(df_single)} images at {IMG_SIZE}×{IMG_SIZE}...')

for _, row in df_single.iterrows():
    img_path = os.path.join(IMAGES_DIR, row['image_id_str'] + '.jpg')

    if not os.path.exists(img_path):
        skipped += 1
        continue

    img = cv2.imread(img_path)
    if img is None:
        skipped += 1
        continue

    orig_h, orig_w = img.shape[:2]

    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_rgb     = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_norm    = (img_rgb / 255.0).astype(np.float32)

    bbox = np.clip([
        row['xmin'] / orig_w,
        row['ymin'] / orig_h,
        row['xmax'] / orig_w,
        row['ymax'] / orig_h
    ], 0.0, 1.0).astype(np.float32)

    X_images.append(img_norm)
    y_labels.append(row['label_enc'])
    y_bboxes.append(bbox)

X_images = np.array(X_images, dtype=np.float32)
y_labels = np.array(y_labels, dtype=np.int32)
y_bboxes = np.array(y_bboxes, dtype=np.float32)

print(f'\n✅ Loaded  : {X_images.shape[0]} images')
print(f'⚠️  Skipped : {skipped}')
print(f'Memory    : {X_images.nbytes / 1e6:.1f} MB')
print(f'Shape → X:{X_images.shape}  y_labels:{y_labels.shape}  y_bboxes:{y_bboxes.shape}')


### 5.3 Visualize Sample Images with Ground Truth Bounding Boxes

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.ravel()
sample_idx = np.random.choice(len(X_images), 10, replace=False)

for i, idx in enumerate(sample_idx):
    img = X_images[idx].copy()
    h, w = img.shape[:2]
    bb   = y_bboxes[idx]

    axes[i].imshow(img)
    rect = patches.Rectangle(
        (bb[0]*w, bb[1]*h), (bb[2]-bb[0])*w, (bb[3]-bb[1])*h,
        linewidth=2, edgecolor='lime', facecolor='none'
    )
    axes[i].add_patch(rect)
    axes[i].set_title(class_names[y_labels[idx]], fontsize=9,
                      color='red', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Images — Ground Truth Bounding Boxes (green)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '02_sample_images_gt.png'), dpi=120)
plt.show()

### 5.4 Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
# ── Stratified 70/15/15 split ─────────────────────────────
X_tr, X_tmp, yl_tr, yl_tmp, yb_tr, yb_tmp = train_test_split(
    X_images, y_labels, y_bboxes,
    test_size=0.30, random_state=42, stratify=y_labels
)
X_val, X_test, yl_val, yl_test, yb_val, yb_test = train_test_split(
    X_tmp, yl_tmp, yb_tmp,
    test_size=0.50, random_state=42, stratify=yl_tmp
)

# One-hot encode class labels
yc_tr   = to_categorical(yl_tr,   NUM_CLASSES)
yc_val  = to_categorical(yl_val,  NUM_CLASSES)
yc_test = to_categorical(yl_test, NUM_CLASSES)

print(f'Train      : {X_tr.shape[0]} images')
print(f'Validation : {X_val.shape[0]} images')
print(f'Test       : {X_test.shape[0]} images')

## 6. Model Architecture

A **lightweight dual-head CNN** using `SeparableConv2D` layers for efficiency.
- **Head 1 – Classification**: softmax probabilities over 11 vehicle classes
- **Head 2 – Bounding Box Regression**: sigmoid-normalized `[xmin, ymin, xmax, ymax]`

In [ ]:
def build_lightweight_detector(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                num_classes=NUM_CLASSES):
    """
    Lightweight dual-head CNN for vehicle detection.
    Outputs:
      - class_output : softmax probabilities over vehicle types
      - bbox_output  : [xmin, ymin, xmax, ymax] normalized to [0,1]
    """
    inputs = Input(shape=input_shape, name='image_input')

    # ── Block 1 ──────────────────────────────────────────
    x = Conv2D(32, (3,3), activation='relu', padding='same', name='conv1')(inputs)
    x = BatchNormalization(name='bn1')(x)
    x = MaxPooling2D((2,2), name='pool1')(x)          # 64→32

    # ── Block 2 (SeparableConv = lightweight) ────────────
    x = SeparableConv2D(64, (3,3), activation='relu', padding='same', name='sep2')(x)
    x = BatchNormalization(name='bn2')(x)
    x = MaxPooling2D((2,2), name='pool2')(x)          # 32→16

    # ── Block 3 ──────────────────────────────────────────
    x = SeparableConv2D(128, (3,3), activation='relu', padding='same', name='sep3')(x)
    x = BatchNormalization(name='bn3')(x)
    x = MaxPooling2D((2,2), name='pool3')(x)          # 16→8

    # ── Block 4 ──────────────────────────────────────────
    x = SeparableConv2D(256, (3,3), activation='relu', padding='same', name='sep4')(x)
    x = BatchNormalization(name='bn4')(x)

    # ── Shared Feature Vector ────────────────────────────
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(256, activation='relu', name='shared_dense')(x)
    x = Dropout(0.4, name='shared_drop')(x)

    # ── Head 1: Classification ───────────────────────────
    cls = Dense(128, activation='relu', name='cls_dense')(x)
    cls = Dropout(0.3, name='cls_drop')(cls)
    cls_out = Dense(num_classes, activation='softmax',
                    name='class_output')(cls)

    # ── Head 2: Bounding Box Regression ──────────────────
    bb = Dense(128, activation='relu', name='bb_dense')(x)
    bb = Dropout(0.3, name='bb_drop')(bb)
    bb_out = Dense(4, activation='sigmoid',
                   name='bbox_output')(bb)
    # sigmoid ensures output is [0,1] (matches normalized bbox targets)

    model = Model(inputs=inputs,
                  outputs=[cls_out, bb_out],
                  name='LightweightVehicleDetector')
    return model


# Build and compile
model = build_lightweight_detector()

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss={
        'class_output': 'categorical_crossentropy',
        'bbox_output' : 'mse'
    },
    loss_weights={
        'class_output': 1.0,
        'bbox_output' : 1.0
    },
    metrics={
        'class_output': 'accuracy',
        'bbox_output' : 'mae'
    }
)

model.summary()

# Count parameters
total_params = model.count_params()
print(f'\nTotal parameters : {total_params:,}')
print(f'Approx model size: {total_params * 4 / 1e6:.2f} MB (float32)')

## 7. Model Training

In [ ]:
# ── Callbacks ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=6,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.4,
        patience=3, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, 'best_model.h5'),
        monitor='val_loss', save_best_only=True, verbose=1
    )
]

print('Training started — CPU-only mode (this will take several minutes)...')
print('EarlyStopping is active — training stops automatically when val_loss stops improving.\n')

history = model.fit(
    X_tr,
    {'class_output': yc_tr, 'bbox_output': yb_tr},
    validation_data=(
        X_val,
        {'class_output': yc_val, 'bbox_output': yb_val}
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training complete!')

### 7.1 Training History Plots

In [ ]:
hist = history.history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Classification Accuracy ───────────────────────────────
axes[0].plot(hist.get('class_output_accuracy', []), label='Train', color='royalblue', lw=2)
axes[0].plot(hist.get('val_class_output_accuracy', []), label='Validation', color='darkorange', lw=2)
axes[0].set_title('Classification Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_ylim([0, 1])

# ── Total Loss ────────────────────────────────────────────
axes[1].plot(hist.get('loss', []), label='Train Loss', color='royalblue', lw=2)
axes[1].plot(hist.get('val_loss', []), label='Val Loss', color='darkorange', lw=2)
axes[1].set_title('Total Loss (Cls + BBox)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

# ── BBox MAE ──────────────────────────────────────────────
axes[2].plot(hist.get('bbox_output_mae', []), label='Train BBox MAE', color='green', lw=2)
axes[2].plot(hist.get('val_bbox_output_mae', []), label='Val BBox MAE', color='red', lw=2)
axes[2].set_title('Bounding Box Regression MAE', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MAE')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Training History — Lightweight CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '03_training_history.png'), dpi=120)
plt.show()

## 8. Model Evaluation

### 8.1 Test Set Metrics

In [ ]:
# ── Load best saved weights ───────────────────────────────
model.load_weights(os.path.join(MODELS_DIR, 'best_model.h5'))

# ── Evaluate ──────────────────────────────────────────────
test_results = model.evaluate(
    X_test,
    {'class_output': yc_test, 'bbox_output': yb_test},
    verbose=0
)

print('\n' + '='*55)
print('TEST SET EVALUATION RESULTS')
print('='*55)
for name, val in zip(model.metrics_names, test_results):
    print(f'  {name:45s}: {val:.4f}')

### 8.2 Classification Report & Confusion Matrix

In [ ]:
# ── Predictions ───────────────────────────────────────────
cls_preds, bb_preds = model.predict(X_test, verbose=0)

y_pred_cls = np.argmax(cls_preds, axis=1)   # Predicted class indices
y_true_cls = yl_test                         # Ground truth class indices

# ── Classification Report ─────────────────────────────────
print('\nCLASSIFICATION REPORT:')
print(classification_report(
    y_true_cls, y_pred_cls,
    target_names=class_names,
    zero_division=0
))

# ── Confusion Matrix ──────────────────────────────────────
cm = confusion_matrix(y_true_cls, y_pred_cls)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5)
plt.title('Confusion Matrix — Vehicle Classification', fontsize=13, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '04_confusion_matrix.png'), dpi=120)
plt.show()

### 8.3 IoU (Intersection over Union) Analysis

In [ ]:
def compute_iou_batch(true_boxes, pred_boxes):
    """Vectorized IoU computation for arrays of normalized boxes."""
    ix1 = np.maximum(true_boxes[:,0], pred_boxes[:,0])
    iy1 = np.maximum(true_boxes[:,1], pred_boxes[:,1])
    ix2 = np.minimum(true_boxes[:,2], pred_boxes[:,2])
    iy2 = np.minimum(true_boxes[:,3], pred_boxes[:,3])

    inter = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)
    area1 = (true_boxes[:,2]-true_boxes[:,0]) * (true_boxes[:,3]-true_boxes[:,1])
    area2 = (pred_boxes[:,2]-pred_boxes[:,0]) * (pred_boxes[:,3]-pred_boxes[:,1])
    union = area1 + area2 - inter

    return np.where(union > 0, inter / union, 0.0)

iou_scores = compute_iou_batch(yb_test, bb_preds)
mean_iou   = np.mean(iou_scores)
iou_at_50  = np.mean(iou_scores >= 0.5)
iou_at_25  = np.mean(iou_scores >= 0.25)

print(f'Bounding Box Evaluation:')
print(f'  Mean IoU (mIoU)       : {mean_iou:.4f}')
print(f'  IoU ≥ 0.50 (standard) : {iou_at_50:.1%}  of test images')
print(f'  IoU ≥ 0.25 (lenient)  : {iou_at_25:.1%}  of test images')

# Plot IoU distribution
plt.figure(figsize=(9, 4))
plt.hist(iou_scores, bins=30, color='teal', edgecolor='black', alpha=0.85)
plt.axvline(0.50, color='red',    ls='--', lw=2, label='IoU=0.50 threshold')
plt.axvline(mean_iou, color='gold', ls='-',  lw=2, label=f'Mean IoU={mean_iou:.3f}')
plt.title('IoU Distribution on Test Set', fontweight='bold')
plt.xlabel('IoU Score')
plt.ylabel('Number of Images')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '05_iou_distribution.png'), dpi=120)
plt.show()

### 8.4 Inference Visualization

In [ ]:
# ── Show 12 test images with ground truth vs predicted ───
# Green solid  = Ground Truth bbox
# Red dashed   = Predicted bbox

n_show = 12
idxs   = np.random.choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for i, idx in enumerate(idxs):
    img  = X_test[idx].copy()
    h, w = img.shape[:2]

    gt_box  = yb_test[idx]
    pd_box  = bb_preds[idx]
    gt_lbl  = class_names[yl_test[idx]]
    pd_lbl  = class_names[y_pred_cls[idx]]
    conf    = cls_preds[idx][y_pred_cls[idx]]
    iou_val = iou_scores[idx]

    axes[i].imshow(img)

    # Ground truth (solid green)
    axes[i].add_patch(patches.Rectangle(
        (gt_box[0]*w, gt_box[1]*h),
        (gt_box[2]-gt_box[0])*w, (gt_box[3]-gt_box[1])*h,
        linewidth=2, edgecolor='lime', facecolor='none', label='GT'
    ))

    # Predicted (red dashed)
    axes[i].add_patch(patches.Rectangle(
        (pd_box[0]*w, pd_box[1]*h),
        (pd_box[2]-pd_box[0])*w, (pd_box[3]-pd_box[1])*h,
        linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
    ))

    correct = (gt_lbl == pd_lbl)
    color   = 'green' if correct else 'red'
    axes[i].set_title(
        f'GT: {gt_lbl}\nPred: {pd_lbl} ({conf:.0%}) | IoU:{iou_val:.2f}',
        fontsize=8, color=color, fontweight='bold'
    )
    axes[i].axis('off')

plt.suptitle(
    'Inference Results  |  🟢 Green=Ground Truth  🔴 Red Dashed=Predicted',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, '06_inference_results.png'), dpi=120)
plt.show()

## 9. Save Model & Class Mapping

In [ ]:
# Save in TF SavedModel format
model.save(os.path.join(MODELS_DIR, 'vehicle_detector_final'))

# Save class mapping as JSON
with open(os.path.join(MODELS_DIR, 'class_names.json'), 'w') as f:
    json.dump({str(i): name for i, name in enumerate(class_names)}, f, indent=2)

print('✅ Model saved to:', os.path.join(MODELS_DIR, 'vehicle_detector_final'))
print('✅ Class map saved to:', os.path.join(MODELS_DIR, 'class_names.json'))
print('\nClass mapping:')
for i, name in enumerate(class_names):
    print(f'  {i}: {name}')

## 10. Project Summary

In [ ]:
print('=' * 65)
print('   CAPSTONE PROJECT COMPLETE — SUMMARY')
print('=' * 65)

print('\n── PART 1: Vehicle Object Detection ──')
print(f'  Architecture    : Lightweight CNN (SeparableConv2D dual-head)')
print(f'  Input Size      : {IMG_SIZE}×{IMG_SIZE} pixels')
print(f'  Vehicle Classes : {NUM_CLASSES} → {class_names}')
print(f'  Outputs         : Vehicle class (softmax) + BBox (sigmoid)')
print(f'  Evaluation      : Accuracy, IoU, MAE, Confusion Matrix')